# ARTGL v2.0: run all experiments on Google Colab

**Before you start:** Runtime → Change runtime type → **L4 GPU** (or T4) → Save. The reported results were computed on an NVIDIA L4.

**On Google Drive**, the folder `MyDrive/datasets` must contain your dataset zips and
`Adaptive-Regularized-v2.0.zip` (version 2.0.6 or newer). A separate `run_all.py` is **not** needed:
the notebook takes it from inside the newest code zip.

Run the cells in order. **If Colab disconnects**, reconnect and run the cells again from the top:
finished training runs are stored on Drive (`MyDrive/datasets/ARTGL_results`) and are skipped.

## 1. Connect Google Drive and pick the newest code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
FOLDER = '/content/drive/MyDrive/datasets'      # change if your folder has another name

import os, glob, zipfile
os.chdir(FOLDER)
print("Files in", FOLDER)
for f in sorted(os.listdir(FOLDER)):
    print("  ", f)

def zip_version(path):
    with zipfile.ZipFile(path) as z:
        name = next((n for n in z.namelist() if n.endswith("/VERSION")), None)
        return z.read(name).decode().strip() if name else "0 (no VERSION file - old)"

def version_key(text):
    try:
        return tuple(int(x) for x in text.split()[0].split("."))
    except ValueError:
        return (0,)

zips = glob.glob(os.path.join(FOLDER, "Adaptive-Regularized-v2*.zip"))
assert zips, "No Adaptive-Regularized-v2.0.zip in the Drive folder - upload it first."
versions = {z: zip_version(z) for z in zips}
for z, v in versions.items():
    print(f"code zip: {os.path.basename(z):45s} version {v}")
best = max(zips, key=lambda z: (version_key(versions[z]), os.path.getmtime(z)))
assert version_key(versions[best]) >= (2, 0, 6), "Upload the newest Adaptive-Regularized-v2.0.zip (2.0.6 or newer)."

# take run_all.py from inside the newest code zip
with zipfile.ZipFile(best) as z:
    member = next(n for n in z.namelist() if n.endswith("/run_all.py"))
    open("/content/run_all.py", "wb").write(z.read(member))
print("\nUsing code", versions[best], "from", os.path.basename(best))

## 2. Check the GPU (must show an NVIDIA L4, Tesla T4 or similar)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Run everything

- Code, extracted data and caches go to Colab's fast local disk (`/content/ARTGL_work`).
- Results, the log and progress markers go to Drive (`ARTGL_results`), so nothing is lost when Colab disconnects.
- `--sensitivity-seeds 0 1 2` runs the long sensitivity sweep with 3 seeds instead of 5 to save GPU time.
  **State this in the paper** (Table 8 caption). Remove it if you have enough GPU time (e.g. Colab Pro).
- Keep this browser tab open; the free version disconnects when idle.

In [ ]:
!python /content/run_all.py --allow-partial-cse --sensitivity-seeds 0 1 2 --work /content/ARTGL_work --results "{FOLDER}/ARTGL_results"

## 4. Look at the results so far (works any time, even before everything is finished)

In [ ]:
!python /content/run_all.py --only report --work /content/ARTGL_work --results "{FOLDER}/ARTGL_results"
report = f"{FOLDER}/ARTGL_results/ALL_RESULTS.txt"
print(open(report, encoding="utf-8").read() if os.path.exists(report) else "No results yet - run cell 3 first.")

## 5. Send back

Download `MyDrive/datasets/ARTGL_results/send_to_claude.zip` (or just `ALL_RESULTS.txt`) from Google Drive and send it to Claude.